# 05. Synthetic-Trace Validation with 100 Experiments

The notebook tests a trace model closer to a real measurement:

\[
T_i[t] = A_i L_i[t-\tau_i] + B_i[t] + U_i[t] + \epsilon_i[t].
\]

`L` is the leakage signal generated from the existing ECDSA model; `A` changes the amplitude, `\tau` introduces a temporal shift, `B(t)` models slow baseline variation, `U(t)` models activity uncorrelated with the nonce, and `\epsilon(t)` is independent Gaussian noise.

The experiment runs exactly 100 independent instances with seeds `1000...1099`. Each instance uses a different ECDSA sample from the project dataset and keeps the components separate for numerical verification.

**Scope of this notebook:** validation of the trace generator and measurement of how much leakage signal remains visible. The results are not treated as ESP32 hardware measurements.

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = next(
    p for p in [Path.cwd(), *Path.cwd().parents]
    if (p / "informatica").is_dir() and (p / "notebooks").is_dir()
)

ECDSA_DIR = PROJECT_ROOT / "informatica" / "ECDSA"
SIM_DIR = PROJECT_ROOT / "informatica" / "simulator"
ARTIFACTS_DIR = PROJECT_ROOT / "informatica" / "artifacts"
RESULTS_DIR = PROJECT_ROOT / "results"
FIGURES_DIR = RESULTS_DIR / "figures"
TABLES_DIR = RESULTS_DIR / "tables"
RAW_DIR = RESULTS_DIR / "raw"

for path in (ECDSA_DIR, SIM_DIR):
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))

for directory in (ARTIFACTS_DIR, FIGURES_DIR, TABLES_DIR, RAW_DIR):
    directory.mkdir(parents=True, exist_ok=True)


In [ ]:
import json
import random
from dataclasses import asdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from dataset_generator3 import CURVE, load_ground_truth, load_public_dataset
from ecdsa_leakage_model4 import generate_clean_leakage
from realistic_trace import RealisticTraceConfig, generate_trace, evaluate_trace


## 1. Experimental Configuration

The values below are simulation parameters, not values claimed to have been measured on an ESP32. They can be changed in a separate sweep. In the main experiment, we retain `trace_length = 256`, `leaked_bits = 12`, and `noise_sigma = 0.15`, so that the leakage portion remains compatible with the existing project.

In [ ]:
config = RealisticTraceConfig(
    trace_length=256,
    noise_sigma=0.15,
    amplitude_mean=1.0,
    amplitude_std=0.08,
    min_amplitude=0.70,
    max_amplitude=1.30,
    max_shift=4,
    baseline_offset_std=0.50,
    baseline_slope_std=0.60,
    baseline_sine_std=0.35,
    unrelated_activity_count=(1, 3),
    unrelated_activity_amplitude=1.50,
    unrelated_activity_width=(2.0, 8.0),
    leakage_alpha=1.0,
    leakage_mode="HW",
    seed=1000,
)

config.validate()
display(pd.DataFrame([
    ["trace_length", config.trace_length, "eșantioane / urmă"],
    ["noise_sigma", config.noise_sigma, "abaterea standard a zgomotului"],
    ["amplitude", f"N({config.amplitude_mean}, {config.amplitude_std**2:.4f}) limitat la [{config.min_amplitude}, {config.max_amplitude}]", "gain al leakage-ului"],
    ["shift", f"[-{config.max_shift}, +{config.max_shift}]", "eșantioane"],
    ["baseline", f"offset={config.baseline_offset_std}, slope={config.baseline_slope_std}, sine={config.baseline_sine_std}", "variație lentă"],
    ["unrelated", f"{config.unrelated_activity_count[0]}...{config.unrelated_activity_count[1]} impulsuri", "activitate independentă de nonce"],
    ["seeds", "1000...1099", "100 experimente"],
], columns=["parametru", "valoare", "rol"]))

## 2. Dataset and Leakage Signal

We use the same `public_dataset.json` and `oracle_dataset.json` from the project. For each `sample_id`, we calculate the clean signal with the existing `generate_clean_leakage()` function. The private key and complete nonce remain oracle data for validation; they do not enter the observable trace.

In [ ]:
public_records = load_public_dataset(ARTIFACTS_DIR / "public_dataset.json")
oracle = load_ground_truth(ARTIFACTS_DIR / "oracle_dataset.json")

if len(public_records) < 100 or len(oracle["samples"]) < 100:
    raise ValueError("Proiectul trebuie să conțină cel puțin 100 de eșantioane pentru acest test.")

clean_by_id = {}
for record in public_records[:100]:
    sample_id = int(record["sample_id"])
    nonce = int(oracle["samples"][str(sample_id)]["nonce_k"])
    clean_by_id[sample_id] = np.asarray(
        generate_clean_leakage(
            nonce,
            config.trace_length,
            mode=config.leakage_mode,
            alpha=config.leakage_alpha,
        ),
        dtype=float,
    )

assert len(clean_by_id) == 100
print(f"Pregătite {len(clean_by_id)} semnale de leakage pentru test.")

## 3. An Individual Experiment: Trace Decomposition

For each trace, we retain the five contributions separately. The algebraic verification is straightforward:

\[
T - B - U - \epsilon = A\,L_{shifted}.
\]

If this identity is not satisfied numerically, the generator has an implementation error.

In [ ]:
example_id = 0
example_seed = config.seed + example_id
example_rng = random.Random(example_seed)
example_components = generate_trace(clean_by_id[example_id], config, example_rng)
example_metrics = evaluate_trace(example_components)

reconstructed = (
    example_components.amplitude_scaled
    + example_components.baseline
    + example_components.unrelated_activity
    + example_components.noise
)

print("Eroare maximă de reconstrucție:", float(np.max(np.abs(example_components.trace - reconstructed))))
print("Amplitudine:", example_components.amplitude)
print("Shift:", example_components.shift)
print("SNR estimat (dB):", example_metrics["snr_db"])


In [ ]:
x = np.arange(config.trace_length)

plt.figure(figsize=(12, 5))
plt.plot(x, example_components.trace, label="trace observat")
plt.plot(x, example_components.amplitude_scaled, label="leakage × amplitudine")
plt.plot(x, example_components.baseline, label="baseline")
plt.plot(x, example_components.unrelated_activity, label="activitate necorelată")
plt.plot(x, example_components.noise, label="zgomot")
plt.xlabel("t")
plt.ylabel("amplitudine")
plt.title("Descompunerea unei urme sintetice")
plt.legend()
plt.tight_layout()
plt.savefig(FIGURES_DIR / "trace-validation-example.png", dpi=160)
plt.show()


## 4. The 100 Experiments

Each experiment has its own seed. Thus, the same trace can be regenerated exactly, and the seed is sufficient to identify the random configuration of amplitude, shift, baseline, uncorrelated activity, and noise.

We measure:

- `snr_db`: the ratio of leakage power after amplitude adjustment to total perturbation power;
- `leakage_correlation`: correlation between the leakage component and the observed trace;
- RMS for baseline, uncorrelated activity, and noise;
- `reconstruction_relative_error`: verifies implementation of the trace equation.

These values test the generator. They are not CNN accuracy and not HNP recovery rate.

In [ ]:
rows = []
traces = []
leakages = []
baselines = []
unrelated = []
noise = []
amplitudes = []
shifts = []

for experiment_id, sample_id in enumerate(list(clean_by_id)[:100], start=1):
    seed = config.seed + experiment_id - 1
    rng = random.Random(seed)
    components = generate_trace(clean_by_id[sample_id], config, rng)
    metrics = evaluate_trace(components)

    rows.append({
        "experiment_id": experiment_id,
        "seed": seed,
        "sample_id": sample_id,
        "amplitude": components.amplitude,
        "shift": components.shift,
        "noise_sigma": config.noise_sigma,
        **metrics,
    })
    traces.append(components.trace)
    leakages.append(components.leakage)
    baselines.append(components.baseline)
    unrelated.append(components.unrelated_activity)
    noise.append(components.noise)
    amplitudes.append(components.amplitude)
    shifts.append(components.shift)

results_df = pd.DataFrame(rows)
results_df.to_csv(TABLES_DIR / "trace-validation-100.csv", index=False)
np.savez_compressed(
    RAW_DIR / "trace-validation-100.npz",
    traces=np.asarray(traces),
    leakages=np.asarray(leakages),
    baselines=np.asarray(baselines),
    unrelated_activity=np.asarray(unrelated),
    noise=np.asarray(noise),
    amplitudes=np.asarray(amplitudes),
    shifts=np.asarray(shifts),
)

metadata = {
    "experiment_count": 100,
    "seed_range": [1000, 1099],
    "trace_length": config.trace_length,
    "leaked_bits": 12,
    "formula": "T[t] = A * L[t - tau] + B[t] + U[t] + epsilon[t]",
    "config": asdict(config),
    "outputs": {
        "csv": str(TABLES_DIR / "trace-validation-100.csv"),
        "npz": str(RAW_DIR / "trace-validation-100.npz"),
        "example_figure": str(FIGURES_DIR / "trace-validation-example.png"),
    },
}
(ARTIFACTS_DIR / "trace_validation_metadata.json").write_text(
    json.dumps(metadata, indent=2), encoding="utf-8"
)

print(f"Au fost rulate {len(results_df)} experimente.")
display(results_df.head(10))


## 5. Integrity Criteria

For this test, the `PASS` conditions are mechanical:

1. exactly 100 results must exist;
2. seeds must be exactly `1000...1099`;
3. amplitude must remain within the configured limits;
4. shift must remain in `[-4, +4]`;
5. reconstruction identity must have practically zero numerical error;
6. each experiment must contain noise and, in the default configuration, uncorrelated activity;
7. runs with the same seeds must produce the same traces.

We do not impose a universal threshold for SNR or correlation. These thresholds depend on the stress level desired for CNN/HNP and must be established separately before a performance benchmark.

In [ ]:
assert len(results_df) == 100
assert results_df["seed"].tolist() == list(range(1000, 1100))
assert results_df["amplitude"].between(config.min_amplitude, config.max_amplitude).all()
assert results_df["shift"].between(-config.max_shift, config.max_shift).all()
assert results_df["reconstruction_relative_error"].max() < 1e-12
assert results_df["noise_rms"].gt(0).all()
assert results_df["unrelated_rms"].ge(0).all()

print("PASS: integritatea celor 100 de experimente este validată.")


## 6. Statistical Summary of the 100 Traces

In [ ]:
summary = pd.DataFrame({
    "metric": [
        "amplitude", "shift", "baseline_rms", "unrelated_rms",
        "noise_rms", "nuisance_rms", "snr_db", "leakage_correlation"
    ],
    "mean": [results_df[c].mean() for c in ["amplitude", "shift", "baseline_rms", "unrelated_rms", "noise_rms", "nuisance_rms", "snr_db", "leakage_correlation"]],
    "median": [results_df[c].median() for c in ["amplitude", "shift", "baseline_rms", "unrelated_rms", "noise_rms", "nuisance_rms", "snr_db", "leakage_correlation"]],
    "min": [results_df[c].min() for c in ["amplitude", "shift", "baseline_rms", "unrelated_rms", "noise_rms", "nuisance_rms", "snr_db", "leakage_correlation"]],
    "max": [results_df[c].max() for c in ["amplitude", "shift", "baseline_rms", "unrelated_rms", "noise_rms", "nuisance_rms", "snr_db", "leakage_correlation"]],
})

display(summary)

In [ ]:
fig = plt.figure(figsize=(11, 4.5))
plt.hist(results_df["snr_db"], bins=12)
plt.xlabel("SNR (dB)")
plt.ylabel("număr de experimente")
plt.title("Distribuția SNR pentru 100 de urme")
plt.tight_layout()
plt.savefig(FIGURES_DIR / "trace-validation-snr.png", dpi=160)
plt.show()

fig = plt.figure(figsize=(11, 4.5))
plt.scatter(results_df["shift"], results_df["leakage_correlation"], alpha=0.8)
plt.xlabel("shift (eșantioane)")
plt.ylabel("corelație cu leakage")
plt.title("Corelația leakage vs. deplasare temporală")
plt.tight_layout()
plt.savefig(FIGURES_DIR / "trace-validation-shift-correlation.png", dpi=160)
plt.show()


## 7. How the Test Relates to the Rest of the Project

This notebook should be viewed as a validation stage for the input data. In the existing project, the CNN receives traces and produces nonce prefixes, and the HNP stage uses those prefixes together with `r`, `s`, and `z`. Therefore, after the generator passes the 100-experiment test, a separate benchmark can compare CNN accuracy for simple traces and for traces with perturbations.

The test above does not modify the `96 / 24 / 40` split in the CNN protocol and does not change the HNP validation criterion. Furthermore, the results remain synthetic until physical ESP32 measurements become available.